# Gold Layer — Retail Sales & Inventory KPIs
Aggregate daily sales by store/category, calculate basket metrics and inventory turnover.

In [ ]:
# Enable V-Order and Optimize Write for read-optimized Gold tables


In [ ]:
from pyspark.sql.functions import (
    col, sum as _sum, avg as _avg, count, countDistinct,
    min as _min, max as _max, current_timestamp,
    round as _round, when
)

In [ ]:
# Daily sales summary by store and product category
df_txn = spark.read.format('delta').table('silver_pos_transactions')
df_products = spark.read.format('delta').table('silver_products')
df_stores = spark.read.format('delta').table('silver_stores')

# Join transactions with dimensions
df_enriched = (
    df_txn
    .join(df_products.select('sku', 'product_name', 'category', 'subcategory', 'brand', 'unit_cost'),
          df_txn.product_id == df_products.sku, 'left')
    .join(df_stores.select('store_id', 'store_name', 'city', 'state', 'region', 'store_format'),
          'store_id', 'left')
)

gold_sales = (
    df_enriched
    .groupBy('transaction_date', 'store_id', 'store_name', 'region',
             'store_format', 'category', 'subcategory')
    .agg(
        countDistinct('transaction_id').alias('transaction_count'),
        count('*').alias('line_item_count'),
        _sum('quantity').alias('total_units_sold'),
        _sum('line_total').alias('total_revenue'),
        _avg('line_total').alias('avg_line_total'),
        _avg('discount_pct').alias('avg_discount_pct'),
        _sum(col('quantity') * col('unit_cost')).alias('total_cost'),
    )
    .withColumn('gross_margin',
                _round(col('total_revenue') - col('total_cost'), 2))
    .withColumn('margin_pct',
                _round(when(col('total_revenue') > 0,
                           col('gross_margin') / col('total_revenue') * 100)
                       .otherwise(0), 2))
    .withColumn('avg_basket_size',
                _round(col('total_revenue') / col('transaction_count'), 2))
    .withColumn('avg_items_per_basket',
                _round(col('total_units_sold') / col('transaction_count'), 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_sales.write.mode('overwrite').format('delta').saveAsTable('gold_sales_daily_summary')
print(f'Gold sales daily summary: {spark.table("gold_sales_daily_summary").count()} rows')

In [ ]:
# Inventory health summary
df_inv = spark.read.format('delta').table('silver_inventory_snapshots')

gold_inventory = (
    df_inv
    .join(df_products.select('sku', 'product_name', 'category'),
          df_inv.product_id == df_products.sku, 'left')
    .groupBy('snapshot_date', 'store_id', 'category')
    .agg(
        _sum('quantity_on_hand').alias('total_on_hand'),
        _sum('quantity_on_order').alias('total_on_order'),
        _sum(col('below_reorder').cast('int')).alias('items_below_reorder'),
        count('*').alias('sku_count'),
    )
    .withColumn('stockout_risk_pct',
                _round(col('items_below_reorder') / col('sku_count') * 100, 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_inventory.write.mode('overwrite').format('delta').saveAsTable('gold_inventory_daily_summary')
print(f'Gold inventory daily summary: {spark.table("gold_inventory_daily_summary").count()} rows')

In [ ]:
# Dimension: Store
dim_store = df_stores.select(
    'store_id', 'store_name', 'city', 'state', 'region', 'store_format'
).distinct()
dim_store.write.mode('overwrite').format('delta').saveAsTable('dim_store')
print(f'dim_store: {spark.table("dim_store").count()} rows')

In [ ]:
# Dimension: Product
dim_product = df_products.select(
    'sku', 'product_name', 'category', 'subcategory', 'brand', 'unit_cost'
).distinct()
dim_product.write.mode('overwrite').format('delta').saveAsTable('dim_product')
print(f'dim_product: {spark.table("dim_product").count()} rows')

In [ ]:
# Gold: Product performance — overall sales aggregated by product
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

gold_product = (
    df_enriched
    .groupBy('sku', 'product_name', 'category', 'subcategory', 'brand')
    .agg(
        _sum('quantity').alias('total_units_sold'),
        _sum('line_total').alias('total_revenue'),
        _sum(col('quantity') * col('unit_cost')).alias('total_cost'),
        countDistinct('transaction_id').alias('transaction_count'),
        countDistinct('store_id').alias('stores_sold_in'),
        _avg('discount_pct').alias('avg_discount_pct'),
    )
    .withColumn('gross_margin', _round(col('total_revenue') - col('total_cost'), 2))
    .withColumn('margin_pct', _round(
        when(col('total_revenue') > 0, col('gross_margin') / col('total_revenue') * 100).otherwise(0), 2))
    .withColumn('revenue_rank', dense_rank().over(Window.orderBy(col('total_revenue').desc())))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_product.write.mode('overwrite').format('delta').saveAsTable('gold_product_performance')
print(f'Gold product performance: {spark.table("gold_product_performance").count()} rows')

In [ ]:
# Gold: Weekly trends — sales aggregated by week for trend analysis
from pyspark.sql.functions import weekofyear, year, concat_ws, lit

gold_weekly = (
    df_enriched
    .withColumn('year', year('transaction_date'))
    .withColumn('week', weekofyear('transaction_date'))
    .withColumn('year_week', concat_ws('-W', col('year').cast('string'),
                                        when(col('week') < 10, concat_ws('', lit('0'), col('week').cast('string')))
                                        .otherwise(col('week').cast('string'))))
    .groupBy('year_week', 'region', 'category')
    .agg(
        _sum('line_total').alias('weekly_revenue'),
        _sum('quantity').alias('weekly_units'),
        countDistinct('transaction_id').alias('weekly_transactions'),
        _avg('discount_pct').alias('avg_discount_pct'),
        _sum(col('quantity') * col('unit_cost')).alias('weekly_cost'),
    )
    .withColumn('weekly_margin', _round(col('weekly_revenue') - col('weekly_cost'), 2))
    .withColumn('weekly_margin_pct', _round(
        when(col('weekly_revenue') > 0, col('weekly_margin') / col('weekly_revenue') * 100).otherwise(0), 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_weekly.write.mode('overwrite').format('delta').saveAsTable('gold_weekly_trends')
print(f'Gold weekly trends: {spark.table("gold_weekly_trends").count()} rows')

In [ ]:
# Optimize Gold tables
spark.sql('OPTIMIZE gold_sales_daily_summary')
spark.sql('OPTIMIZE gold_inventory_daily_summary')
spark.sql('OPTIMIZE dim_store')
spark.sql('OPTIMIZE dim_product')
spark.sql('OPTIMIZE gold_product_performance')
spark.sql('OPTIMIZE gold_weekly_trends')
print('All Gold tables optimized')